# First Match ≠ Right Match

## Step 1: Load documents and create metadata-aware chunks

We will use the actual EU AI Act PDF and podcast transcript in `data/`. PDF pages are extracted independently so their page numbers remain attached to each chunk. The transcript has no page numbers, so its page metadata is `None`. Chunks use a simple word window with overlap; this keeps the first step easy to inspect and does not rely on an embedding model's tokenizer.

In [1]:
from pathlib import Path
import re

import pymupdf

DATA_DIR = Path("data")
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"
TRANSCRIPT_PATH = DATA_DIR / "podcast.txt"

for path in (PDF_PATH, TRANSCRIPT_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Expected source document at {path.resolve()}")


Each loaded document has text plus metadata. The PDF loader records one-based PDF page numbers. The transcript is a single source with no meaningful page number.

In [2]:
def load_documents() -> list[dict]:
    """Load PDF pages and the transcript as text records with source metadata."""
    documents = []

    with pymupdf.open(PDF_PATH) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if text:
                documents.append(
                    {
                        "text": text,
                        "metadata": {
                            "source": str(PDF_PATH),
                            "document_type": "pdf",
                            "page": page_number,
                        },
                    }
                )

    transcript_text = TRANSCRIPT_PATH.read_text(encoding="utf-8").strip()
    if transcript_text:
        documents.append(
            {
                "text": transcript_text,
                "metadata": {
                    "source": str(TRANSCRIPT_PATH),
                    "document_type": "transcript",
                    "page": None,
                },
            }
        )

    return documents


documents = load_documents()
print(f"Loaded {len(documents)} text records")
print("First PDF page metadata:", documents[0]["metadata"])
print("Transcript metadata:", documents[-1]["metadata"])


Loaded 145 text records
First PDF page metadata: {'source': 'data/eu_ai_act.pdf', 'document_type': 'pdf', 'page': 1}
Transcript metadata: {'source': 'data/podcast.txt', 'document_type': 'transcript', 'page': None}


Chunk each source record separately, preserving its metadata and adding a stable chunk index. Overlap carries nearby context across chunk boundaries. `page` stays `None` for transcript chunks rather than implying a page that the source does not have.

In [3]:
def chunk_text(text: str, chunk_size: int = 250, overlap: int = 40) -> list[str]:
    """Split text into overlapping chunks measured in whitespace-separated words."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size")

    words = re.findall(r"\S+", text)
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if chunk_words:
            chunks.append(" ".join(chunk_words))

    return chunks


def create_chunks(
    documents: list[dict], chunk_size: int = 250, overlap: int = 40
) -> list[dict]:
    """Create chunks while retaining each document's source and page metadata."""
    chunks = []

    for document in documents:
        for chunk_index, text in enumerate(
            chunk_text(document["text"], chunk_size, overlap)
        ):
            chunks.append(
                {
                    "text": text,
                    "metadata": {
                        **document["metadata"],
                        "chunk_index": chunk_index,
                    },
                }
            )

    return chunks


chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks")
print("First chunk metadata:", chunks[0]["metadata"])
print("Last chunk metadata:", chunks[-1]["metadata"])
print("First chunk preview:", chunks[0]["text"][:300])


Created 516 chunks
First chunk metadata: {'source': 'data/eu_ai_act.pdf', 'document_type': 'pdf', 'page': 1, 'chunk_index': 0}
Last chunk metadata: {'source': 'data/podcast.txt', 'document_type': 'transcript', 'page': None, 'chunk_index': 13}
First chunk preview: REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (


Quick checks: every chunk has a source and chunk index, PDF chunks have a page number, and transcript chunks explicitly have no page number.

In [4]:
assert chunks
assert all(chunk["metadata"]["source"] for chunk in chunks)
assert all("chunk_index" in chunk["metadata"] for chunk in chunks)
assert all(
    chunk["metadata"]["page"] is not None
    for chunk in chunks
    if chunk["metadata"]["document_type"] == "pdf"
)
assert all(
    chunk["metadata"]["page"] is None
    for chunk in chunks
    if chunk["metadata"]["document_type"] == "transcript"
)

print("Step 1 checks passed")


Step 1 checks passed


## Step 2: Embed chunks and retrieve by cosine similarity

A sentence-transformer maps each chunk to a numeric vector. The same model embeds the query; cosine similarity ranks chunks by the angle between their vectors. We keep the original chunk text and metadata alongside each score so the ranking can be inspected and compared with a reranker later. The model runs on CPU here to keep execution predictable across notebook environments.

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
import numpy as np

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
embedding_model = SentenceTransformer(
    MODEL_NAME, device="cpu", revision=MODEL_REVISION
)

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f"Embedded {len(chunk_embeddings)} chunks")
print(f"Embedding dimensions: {chunk_embeddings.shape[1]}")


W1028 23:01:17.803000 57362 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Embedded 516 chunks
Embedding dimensions: 384


The search function embeds one query, calculates cosine similarity against every chunk vector, then returns the highest-scoring chunks with their original metadata and text. Similarity is a ranking signal, not a relevance label.

In [6]:
def filter_chunk_indices(
    *,
    source: str | None = None,
    document_type: str | None = None,
    page_range: tuple[int, int] | None = None,
) -> list[int]:
    """Return chunk positions matching the requested source metadata."""
    if document_type not in {None, "pdf", "transcript"}:
        raise ValueError("document_type must be 'pdf', 'transcript', or None")
    if page_range is not None:
        if len(page_range) != 2:
            raise ValueError("page_range must contain an inclusive start and end")
        start_page, end_page = page_range
        if start_page < 1 or end_page < start_page:
            raise ValueError("page_range must be a valid inclusive one-based range")

    matching_indices = []
    for index, chunk in enumerate(chunks):
        metadata = chunk["metadata"]
        if source is not None and metadata["source"] != source:
            continue
        if document_type is not None and metadata["document_type"] != document_type:
            continue
        if page_range is not None:
            page_number = metadata["page"]
            if page_number is None or not start_page <= page_number <= end_page:
                continue
        matching_indices.append(index)

    return matching_indices


def filter_chunks(
    source: str | None = None,
    document_type: str | None = None,
    page_range: tuple[int, int] | None = None,
) -> list[dict]:
    """Return chunks matching source, document type, and inclusive PDF pages."""
    matching_indices = filter_chunk_indices(
        source=source, document_type=document_type, page_range=page_range
    )
    return [chunks[index] for index in matching_indices]


def search_chunks(
    query: str,
    top_k: int = 5,
    *,
    source: str | None = None,
    document_type: str | None = None,
    page_range: tuple[int, int] | None = None,
) -> list[dict]:
    """Search by cosine similarity, optionally restricting source metadata."""
    if not query.strip():
        raise ValueError("query must not be empty")
    if type(top_k) is not int or top_k <= 0:
        raise ValueError("top_k must be greater than zero")

    eligible_indices = filter_chunk_indices(
        source=source, document_type=document_type, page_range=page_range
    )
    if not eligible_indices:
        return []

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )
    similarities = cosine_similarity(
        query_embedding, chunk_embeddings[eligible_indices]
    )[0]
    ranked_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for rank, filtered_index in enumerate(ranked_indices, start=1):
        chunk_index = eligible_indices[int(filtered_index)]
        chunk = chunks[int(chunk_index)]
        text = chunk["text"]
        results.append(
            {
                "rank": rank,
                "similarity_score": float(similarities[int(filtered_index)]),
                "metadata": chunk["metadata"],
                "excerpt": text[:500],
                "text": text,
            }
        )

    return results


TEST_QUERY = (
    "What is the difference between the seven ethical requirements for "
    "trustworthy AI and the legally binding requirements for high-risk AI "
    "systems under the EU AI Act?"
)
baseline_results = search_chunks(TEST_QUERY, top_k=5)


In [7]:
for result in baseline_results:
    metadata = result["metadata"]
    print(
        f"Rank {result['rank']} | "
        f"cosine similarity {result['similarity_score']:.4f} | "
        f"source: {metadata['source']} | page: {metadata['page']}"
    )
    print(result["excerpt"].replace("\n", " "))
    print()

RETRIEVAL_RESULTS_PATH = Path("retrieval_results.json")
retrieval_evidence = {
    "query": TEST_QUERY,
    "method": "sentence-transformer embeddings with cosine similarity",
    "model": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "chunk_count": len(chunks),
    "embedding_dimension": int(chunk_embeddings.shape[1]),
    "results": baseline_results,
}
RETRIEVAL_RESULTS_PATH.write_text(
    json.dumps(retrieval_evidence, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved baseline retrieval evidence to {RETRIEVAL_RESULTS_PATH}")


Rank 1 | cosine similarity 0.7654 | source: data/eu_ai_act.pdf | page: 41
mutandis, without prejudice to more specific procedural rights provided for by this Regulation. (165) The development of AI systems other than high-risk AI systems in accordance with the requirements of this Regulation may lead to a larger uptake of ethical and trustworthy AI in the Union. Providers of AI systems that are not high-risk should be encouraged to create codes of conduct, including related governance mechanisms, intended to foster the voluntary application of some or all of the manda

Rank 2 | cosine similarity 0.7621 | source: data/eu_ai_act.pdf | page: 8
(27) While the risk-based approach is the basis for a proportionate and effective set of binding rules, it is important to recall the 2019 Ethics guidelines for trustworthy AI developed by the independent AI HLEG appointed by the Commission. In those guidelines, the AI HLEG developed seven non-binding ethical principles for AI which are intended to 

## Step 3: Add a relevance score and compare rankings

We first retrieve 20 candidates with the same cosine search. The optional OpenAI scorer is used only when `USE_LLM_RELEVANCE_SCORER` is explicitly enabled and `OPENAI_API_KEY` is set. Otherwise, a deterministic keyword baseline is used and labeled as such; it is a transparent heuristic, not an LLM score or human relevance judgment. In that baseline, 5 means the passage explicitly contrasts the seven non-binding ethical principles with legally binding requirements; 4 means it makes that contrast or contains a direct, cited provider/deployer duty for high-risk AI; 3 means it covers high-risk legal evidence with an Article reference or legal language; 2 means it is specifically about one topic; 1 is an incidental article/requirement match; and 0 means no configured evidence signal matched. We combine min-max-normalized cosine similarity and the relevance score divided by 5 with equal weights. The combined score only reranks these 20 candidates.

In [8]:
import os
import urllib.request

USE_LLM_RELEVANCE_SCORER = False

def score_with_openai(query: str, candidates: list[dict], api_key: str) -> dict:
    """Ask an OpenAI-compatible chat-completions API to score each candidate."""
    model_name = os.getenv("OPENAI_RELEVANCE_MODEL", "gpt-4o-mini")
    candidate_text = [
        {"rank": candidate["rank"], "text": candidate["text"]}
        for candidate in candidates
    ]
    request_body = {
        "model": model_name,
        "temperature": 0,
        "response_format": {"type": "json_object"},
        "messages": [
            {
                "role": "system",
                "content": (
                    "Score each supplied chunk's direct usefulness for answering "
                    "the query from 0 to 5 (integer). Use only the chunk text. "
                    "Distinguish ethical guidelines for trustworthy AI from "
                    "legally binding EU AI Act requirements for high-risk AI. "
                    "Prefer actual legal requirements and article references "
                    "when present. Do not invent evidence or references. "
                    "Return JSON with a 'scores' list; each item must contain "
                    "the supplied rank, an integer score, and a short rationale."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    {"query": query, "candidates": candidate_text},
                    ensure_ascii=False,
                ),
            },
        ],
    }
    request = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=json.dumps(request_body).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    with urllib.request.urlopen(request, timeout=60) as response:
        response_data = json.loads(response.read().decode("utf-8"))
    score_data = json.loads(response_data["choices"][0]["message"]["content"])

    expected_ranks = {candidate["rank"] for candidate in candidates}
    items = score_data.get("scores")
    if not isinstance(items, list) or len(items) != len(expected_ranks):
        raise ValueError("The LLM response must score every candidate exactly once")

    scores = {}
    for item in items:
        rank = item.get("rank")
        score = item.get("score")
        rationale = item.get("rationale")
        if rank not in expected_ranks or rank in scores:
            raise ValueError(f"Unexpected or duplicate candidate rank: {rank}")
        if type(score) is not int or not 0 <= score <= 5:
            raise ValueError(f"Invalid relevance score for candidate {rank}: {score}")
        if not isinstance(rationale, str) or not rationale.strip():
            raise ValueError(f"Missing rationale for candidate {rank}")
        scores[rank] = {"score": score, "rationale": rationale.strip()}

    if set(scores) != expected_ranks:
        raise ValueError("The LLM response did not include every candidate rank")
    return {"scores": scores, "model": model_name}


def score_with_keyword_baseline(chunk: dict) -> dict:
    """Apply a fixed evidence-signal rubric; this is not an LLM judgment."""
    text = chunk["text"].lower()
    has_ethics_guidance = any(
        term in text
        for term in (
            "ethics guidelines for trustworthy ai",
            "ethical requirements",
            "seven key requirements",
            "seven non-binding ethical principles",
            "ai hleg",
        )
    )
    has_seven_principles = (
        "seven" in text and has_ethics_guidance
    )
    has_nonbinding_guidance = (
        has_ethics_guidance and "non-binding" in text
    )
    has_legal_binding_language = any(
        term in text
        for term in (
            "legally binding",
            "binding requirements",
            "binding rules",
        )
    )
    has_high_risk = "high-risk ai" in text or "high risk ai" in text
    has_legal_signal = any(
        term in text
        for term in (
            "obligations",
            "shall ",
            "must ",
            "requirements for high-risk",
            "requirements for high risk",
        )
    )
    has_article_reference = bool(
        re.search(r"\barticle\s+\d+\b", text, flags=re.IGNORECASE)
    )
    explicit_contrast = (
        has_nonbinding_guidance and has_legal_binding_language
    )
    cited_high_risk_obligation = (
        has_high_risk and has_legal_signal and has_article_reference
    )
    direct_provider_duty = bool(
        re.search(
            r"\b(?:providers|deployers) of high-risk ai systems shall\s*:",
            text,
        )
    )

    if explicit_contrast and has_seven_principles:
        score = 5
        reason = "explicitly contrasts seven non-binding ethical principles with binding requirements"
    elif explicit_contrast or direct_provider_duty:
        score = 4
        reason = (
            "contrasts ethical guidance with binding requirements"
            if explicit_contrast
            else "contains a cited direct provider/deployer duty for high-risk AI"
        )
    elif has_ethics_guidance and has_high_risk and has_legal_signal:
        score = 3
        reason = "covers ethical guidance and high-risk AI with legal language"
    elif cited_high_risk_obligation:
        score = 3
        reason = "contains Article-referenced high-risk legal evidence"
    elif has_high_risk and has_legal_signal:
        score = 3
        reason = "covers high-risk AI and legal requirements or obligations"
    elif has_ethics_guidance or has_high_risk:
        score = 2
        reason = "specifically covers one of the two topics"
    elif has_article_reference and has_legal_signal:
        score = 1
        reason = "contains an Article reference and legal language"
    else:
        score = 0
        reason = "no configured relevance signals matched"

    return {"score": score, "rationale": reason}


def score_relevance(query: str, candidates: list[dict]) -> dict:
    """Use the opt-in LLM when keyed; otherwise use the labeled heuristic."""
    api_key = os.getenv("OPENAI_API_KEY")
    if USE_LLM_RELEVANCE_SCORER:
        if not api_key:
            raise RuntimeError(
                "Set OPENAI_API_KEY to opt in to LLM relevance scoring"
            )
        llm_result = score_with_openai(query, candidates, api_key)
        return {
            "method": "openai_llm",
            "model": llm_result["model"],
            "scores": llm_result["scores"],
        }

    return {
        "method": "deterministic_keyword_baseline",
        "model": None,
        "scores": {
            candidate["rank"]: score_with_keyword_baseline(candidate)
            for candidate in candidates
        },
    }


top_20_candidates = search_chunks(TEST_QUERY, top_k=20)
baseline_file_evidence = json.loads(
    RETRIEVAL_RESULTS_PATH.read_text(encoding="utf-8")
)
original_top_5 = baseline_file_evidence["results"]

def evidence_key(result: dict) -> tuple:
    metadata = result["metadata"]
    return (
        metadata["source"],
        metadata["page"],
        metadata["chunk_index"],
    )

if [evidence_key(item) for item in top_20_candidates[:5]] != [
    evidence_key(item) for item in original_top_5
]:
    raise ValueError("The saved baseline top five do not match the current search")

relevance_scoring = score_relevance(TEST_QUERY, top_20_candidates)
score_by_rank = relevance_scoring["scores"]
similarity_scores = [item["similarity_score"] for item in top_20_candidates]
minimum_similarity = min(similarity_scores)
maximum_similarity = max(similarity_scores)
similarity_range = maximum_similarity - minimum_similarity

scored_candidates = []
for candidate in top_20_candidates:
    original_rank = candidate["rank"]
    relevance = score_by_rank[original_rank]
    normalized_similarity = (
        (candidate["similarity_score"] - minimum_similarity) / similarity_range
        if similarity_range
        else 0.0
    )
    normalized_relevance = relevance["score"] / 5
    scored_candidates.append(
        {
            **candidate,
            "original_rank": original_rank,
            "relevance_score": relevance["score"],
            "relevance_rationale": relevance["rationale"],
            "normalized_similarity": normalized_similarity,
            "normalized_relevance": normalized_relevance,
            "combined_score": (
                0.5 * normalized_similarity + 0.5 * normalized_relevance
            ),
        }
    )

combined_ranking = sorted(
    scored_candidates,
    key=lambda item: (-item["combined_score"], item["original_rank"]),
)
for final_rank, candidate in enumerate(combined_ranking, start=1):
    candidate["final_rank"] = final_rank

print("Scoring method:", relevance_scoring["method"])
if relevance_scoring["model"]:
    print("Scoring model:", relevance_scoring["model"])
print("Relevance scale: 0 (not useful) to 5 (directly useful)")
print("Combined score: 0.5 * normalized cosine + 0.5 * (relevance / 5)")


Scoring method: deterministic_keyword_baseline
Relevance scale: 0 (not useful) to 5 (directly useful)
Combined score: 0.5 * normalized cosine + 0.5 * (relevance / 5)


In [9]:
def display_ranking(title: str, results: list[dict], rank_field: str) -> None:
    """Print ranked evidence and the scores needed to compare both rankings."""
    print(f"\n{title}")
    for result in results:
        metadata = result["metadata"]
        print(
            f"Rank {result[rank_field]} | "
            f"source: {metadata['source']} | page: {metadata['page']} | "
            f"cosine: {result['similarity_score']:.4f} | "
            f"relevance: {result['relevance_score']}/5 | "
            f"combined: {result['combined_score']:.4f}"
        )
        print(result["excerpt"].replace("\n", " "))


original_top_5_scored = sorted(
    scored_candidates, key=lambda item: item["original_rank"]
)[:5]
relevance_scored_top_5 = combined_ranking[:5]
display_ranking("Original cosine top five", original_top_5_scored, "original_rank")
display_ranking("Combined relevance-scored top five", relevance_scored_top_5, "final_rank")

RELEVANCE_RESULTS_PATH = Path("relevance_scoring_results.json")
relevance_evidence = {
    "query": TEST_QUERY,
    "candidate_count": len(top_20_candidates),
    "baseline_method": "sentence-transformer embeddings with cosine similarity",
    "embedding_model": MODEL_NAME,
    "embedding_model_revision": MODEL_REVISION,
    "relevance_scoring_method": relevance_scoring["method"],
    "relevance_scoring_model": relevance_scoring["model"],
    "relevance_scale": "integer from 0 (not useful) to 5 (directly useful)",
    "combined_score_formula": (
        "0.5 * min-max-normalized cosine similarity within the top 20 "
        "+ 0.5 * (relevance score / 5)"
    ),
    "comparison": {
        "original_cosine_top_5": original_top_5_scored,
        "relevance_scored_top_5": relevance_scored_top_5,
    },
    "scored_candidates": combined_ranking,
}
RELEVANCE_RESULTS_PATH.write_text(
    json.dumps(relevance_evidence, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\nSaved comparison and evidence to {RELEVANCE_RESULTS_PATH}")



Original cosine top five
Rank 1 | source: data/eu_ai_act.pdf | page: 41 | cosine: 0.7654 | relevance: 2/5 | combined: 0.7000
mutandis, without prejudice to more specific procedural rights provided for by this Regulation. (165) The development of AI systems other than high-risk AI systems in accordance with the requirements of this Regulation may lead to a larger uptake of ethical and trustworthy AI in the Union. Providers of AI systems that are not high-risk should be encouraged to create codes of conduct, including related governance mechanisms, intended to foster the voluntary application of some or all of the manda
Rank 2 | source: data/eu_ai_act.pdf | page: 8 | cosine: 0.7621 | relevance: 5/5 | combined: 0.9812
(27) While the risk-based approach is the basis for a proportionate and effective set of binding rules, it is important to recall the 2019 Ethics guidelines for trustworthy AI developed by the independent AI HLEG appointed by the Commission. In those guidelines, the AI HLEG

## Step 4: Rerank the baseline candidates with a cross-encoder

A cross-encoder reads the query and a candidate chunk together, then emits one raw relevance score for that pair. We apply it only to the same 20 cosine-retrieved candidates. The raw cross-encoder scores are not calibrated probabilities and are meaningful only for ordering within this reranking run; do not compare their numeric scale with cosine or combined relevance scores. In this environment, loading the cached safetensors checkpoint produced non-finite outputs on CPU, so the model is loaded from its PyTorch checkpoint instead and every score is checked before ranking.

In [10]:
from sentence_transformers import CrossEncoder

CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CROSS_ENCODER_REVISION = "233902d25c440f23af6f7d6e94d2946bac0bee0a"
cross_encoder = CrossEncoder(
    CROSS_ENCODER_MODEL_NAME,
    revision=CROSS_ENCODER_REVISION,
    device="cpu",
    model_kwargs={"use_safetensors": False},
)

baseline_ranking = sorted(scored_candidates, key=lambda item: item["original_rank"])
relevance_ranking = sorted(scored_candidates, key=lambda item: item["final_rank"])
query_chunk_pairs = [
    (TEST_QUERY, candidate["text"]) for candidate in baseline_ranking
]
cross_encoder_scores = cross_encoder.predict(
    query_chunk_pairs,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
)

if len(cross_encoder_scores) != len(baseline_ranking):
    raise ValueError("The cross-encoder did not score every baseline candidate")
if not np.isfinite(cross_encoder_scores).all():
    raise ValueError("The cross-encoder returned a non-finite score")

for candidate, score in zip(baseline_ranking, cross_encoder_scores):
    candidate["cross_encoder_score"] = float(score)

cross_encoder_ranking = sorted(
    baseline_ranking,
    key=lambda item: (-item["cross_encoder_score"], item["original_rank"]),
)
for cross_rank, candidate in enumerate(cross_encoder_ranking, start=1):
    candidate["cross_encoder_rank"] = cross_rank

print(f"Cross-encoder scored {len(cross_encoder_ranking)} candidate chunks")
print("Cross-encoder scores are raw ranking scores, not probabilities.")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Cross-encoder scored 20 candidate chunks
Cross-encoder scores are raw ranking scores, not probabilities.


In [11]:
def display_method_top_five(
    title: str, ranking: list[dict], rank_field: str, score_field: str, score_label: str
) -> None:
    """Display one method's ranking without mixing its score scale with others."""
    print(f"\n{title}")
    for candidate in ranking[:5]:
        metadata = candidate["metadata"]
        print(
            f"Rank {candidate[rank_field]} | {metadata['source']} | "
            f"page {metadata['page']} | {score_label}: "
            f"{candidate[score_field]:.4f}"
        )
        if title == "Step 3 relevance scoring":
            print(f"Relevance score: {candidate['relevance_score']}/5")
        print(candidate["excerpt"].replace("\n", " "))


display_method_top_five(
    "Baseline cosine similarity",
    baseline_ranking,
    "original_rank",
    "similarity_score",
    "cosine similarity",
)
display_method_top_five(
    "Step 3 relevance scoring",
    relevance_ranking,
    "final_rank",
    "combined_score",
    "combined score",
)
display_method_top_five(
    "Cross-encoder reranking (raw score; not a probability)",
    cross_encoder_ranking,
    "cross_encoder_rank",
    "cross_encoder_score",
    "cross-encoder score",
)

def find_evidence_chunk(predicate, description: str) -> dict:
    matches = [candidate for candidate in scored_candidates if predicate(candidate["text"].lower())]
    if not matches:
        raise ValueError(f"Could not find source evidence for: {description}")
    return matches[0]


ethical_comparison_chunk = find_evidence_chunk(
    lambda text: "seven non-binding ethical principles" in text
    and "legally binding requirements" in text,
    "the seven non-binding ethical principles and binding requirements",
)
article_16_chunk = find_evidence_chunk(
    lambda text: "article 16 obligations of providers of high-risk ai systems" in text
    and "providers of high-risk ai systems shall:" in text,
    "Article 16 provider obligations for high-risk AI systems",
)

def summarize_evidence_position(candidate: dict, evidence_type: str) -> dict:
    return {
        "evidence_type": evidence_type,
        "metadata": candidate["metadata"],
        "original_cosine_rank": candidate["original_rank"],
        "relevance_scored_rank": candidate["final_rank"],
        "cross_encoder_rank": candidate["cross_encoder_rank"],
        "in_top_5": {
            "baseline_cosine": candidate["original_rank"] <= 5,
            "relevance_scoring": candidate["final_rank"] <= 5,
            "cross_encoder": candidate["cross_encoder_rank"] <= 5,
        },
        "excerpt": candidate["excerpt"],
    }


evidence_positions = [
    summarize_evidence_position(
        ethical_comparison_chunk,
        "seven non-binding ethical principles versus legally binding requirements",
    ),
    summarize_evidence_position(
        article_16_chunk,
        "Article 16 high-risk AI provider duties",
    ),
]
print("\nEvidence movement across methods:")
for evidence in evidence_positions:
    print(
        f"{evidence['evidence_type']}: baseline rank "
        f"{evidence['original_cosine_rank']}, relevance rank "
        f"{evidence['relevance_scored_rank']}, cross-encoder rank "
        f"{evidence['cross_encoder_rank']}"
    )

RERANKING_RESULTS_PATH = Path("reranking_results.json")
reranking_evidence = {
    "query": TEST_QUERY,
    "candidate_count": len(baseline_ranking),
    "embedding_model": MODEL_NAME,
    "embedding_model_revision": MODEL_REVISION,
    "cross_encoder_model": CROSS_ENCODER_MODEL_NAME,
    "cross_encoder_revision": CROSS_ENCODER_REVISION,
    "cross_encoder_device": "cpu",
    "cross_encoder_score_note": (
        "Raw model scores are not calibrated probabilities and should not be "
        "compared numerically with cosine similarity or combined relevance scores."
    ),
    "rankings": {
        "baseline_cosine": baseline_ranking,
        "step_3_relevance_scoring": relevance_ranking,
        "cross_encoder": cross_encoder_ranking,
    },
    "evidence_positions": evidence_positions,
}
RERANKING_RESULTS_PATH.write_text(
    json.dumps(reranking_evidence, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\nSaved all candidate scores and rankings to {RERANKING_RESULTS_PATH}")



Baseline cosine similarity
Rank 1 | data/eu_ai_act.pdf | page 41 | cosine similarity: 0.7654
mutandis, without prejudice to more specific procedural rights provided for by this Regulation. (165) The development of AI systems other than high-risk AI systems in accordance with the requirements of this Regulation may lead to a larger uptake of ethical and trustworthy AI in the Union. Providers of AI systems that are not high-risk should be encouraged to create codes of conduct, including related governance mechanisms, intended to foster the voluntary application of some or all of the manda
Rank 2 | data/eu_ai_act.pdf | page 8 | cosine similarity: 0.7621
(27) While the risk-based approach is the basis for a proportionate and effective set of binding rules, it is important to recall the 2019 Ethics guidelines for trustworthy AI developed by the independent AI HLEG appointed by the Commission. In those guidelines, the AI HLEG developed seven non-binding ethical principles for AI which are i

## Step 5: Filter retrieval by source metadata

Metadata filters can restrict retrieval to an exact source, document type, and inclusive one-based PDF page range. Filtering happens before similarity ranking; transcript chunks have no page and are excluded from page-range filters. With no filters, baseline retrieval behavior is unchanged.

In [12]:
def filter_chunk_indices(
    *,
    source: str | None = None,
    document_type: str | None = None,
    page_range: tuple[int, int] | None = None,
) -> list[int]:
    """Return chunk positions matching the requested source metadata."""
    if document_type not in {None, "pdf", "transcript"}:
        raise ValueError("document_type must be 'pdf', 'transcript', or None")
    if page_range is not None:
        if len(page_range) != 2:
            raise ValueError("page_range must contain an inclusive start and end")
        start_page, end_page = page_range
        if type(start_page) is not int or type(end_page) is not int:
            raise ValueError("page_range boundaries must be integers")
        if start_page < 1 or end_page < start_page:
            raise ValueError("page_range must be a valid inclusive one-based range")

    matching_indices = []
    for index, chunk in enumerate(chunks):
        metadata = chunk["metadata"]
        if source is not None and metadata["source"] != source:
            continue
        if document_type is not None and metadata["document_type"] != document_type:
            continue
        if page_range is not None:
            page_number = metadata["page"]
            if page_number is None or not start_page <= page_number <= end_page:
                continue
        matching_indices.append(index)

    return matching_indices


def filter_chunks(
    *,
    source: str | None = None,
    document_type: str | None = None,
    page_range: tuple[int, int] | None = None,
) -> list[dict]:
    """Return chunks matching source, document type, and inclusive PDF pages."""
    matching_indices = filter_chunk_indices(
        source=source, document_type=document_type, page_range=page_range
    )
    return [chunks[index] for index in matching_indices]


pdf_chunks = filter_chunks(document_type="pdf")
article_page_chunks = filter_chunks(
    source=str(PDF_PATH), document_type="pdf", page_range=(56, 62)
)
podcast_chunks = filter_chunks(
    source=str(TRANSCRIPT_PATH), document_type="transcript"
)
page_8_chunks = filter_chunks(
    source=str(PDF_PATH), document_type="pdf", page_range=(8, 8)
)

assert len(pdf_chunks) == 502
assert article_page_chunks
assert podcast_chunks
assert page_8_chunks
assert all(chunk["metadata"]["document_type"] == "pdf" for chunk in pdf_chunks)
assert all(
    56 <= chunk["metadata"]["page"] <= 62 for chunk in article_page_chunks
)
assert all(chunk["metadata"]["page"] is None for chunk in podcast_chunks)
assert all(chunk["metadata"]["page"] == 8 for chunk in page_8_chunks)

filtered_pdf_results = search_chunks(
    TEST_QUERY, top_k=5, source=str(PDF_PATH), document_type="pdf"
)
filtered_article_results = search_chunks(
    TEST_QUERY,
    top_k=len(article_page_chunks),
    source=str(PDF_PATH),
    document_type="pdf",
    page_range=(56, 62),
)
filtered_podcast_results = search_chunks(
    TEST_QUERY,
    top_k=5,
    source=str(TRANSCRIPT_PATH),
    document_type="transcript",
)
filtered_page_8_results = search_chunks(
    TEST_QUERY,
    top_k=len(page_8_chunks),
    source=str(PDF_PATH),
    page_range=(8, 8),
)
assert all(result["metadata"]["document_type"] == "pdf" for result in filtered_pdf_results)
assert all(56 <= result["metadata"]["page"] <= 62 for result in filtered_article_results)
assert all(result["metadata"]["source"] == str(TRANSCRIPT_PATH) for result in filtered_podcast_results)
assert all(result["metadata"]["page"] is None for result in filtered_podcast_results)
assert all(result["metadata"]["page"] == 8 for result in filtered_page_8_results)
assert search_chunks(TEST_QUERY, source="data/not_a_source.txt") == []

try:
    filter_chunk_indices(page_range=(62, 56))
except ValueError:
    pass
else:
    raise AssertionError("An inverted page range must raise ValueError")

print(f"PDF-only chunks: {len(pdf_chunks)}")
print(f"Chunks on PDF pages 56-62: {len(article_page_chunks)}")
print(f"Transcript-only chunks: {len(podcast_chunks)}")
print("Metadata filter checks passed")

baseline_evidence = json.loads(
    RETRIEVAL_RESULTS_PATH.read_text(encoding="utf-8")
)
baseline_evidence["metadata_filter_examples"] = {
    "pdf_only_top_5": {
        "filters": {"source": str(PDF_PATH), "document_type": "pdf"},
        "results": filtered_pdf_results,
    },
    "high_risk_requirements_pages_56_to_62": {
        "filters": {
            "source": str(PDF_PATH),
            "document_type": "pdf",
            "page_range": [56, 62],
        },
        "matching_chunk_count": len(article_page_chunks),
        "results": filtered_article_results,
    },
    "podcast_only_top_5": {
        "filters": {
            "source": str(TRANSCRIPT_PATH),
            "document_type": "transcript",
        },
        "results": filtered_podcast_results,
    },
    "pdf_page_8_only": {
        "filters": {
            "source": str(PDF_PATH),
            "document_type": "pdf",
            "page_range": [8, 8],
        },
        "results": filtered_page_8_results,
    },
}
RETRIEVAL_RESULTS_PATH.write_text(
    json.dumps(baseline_evidence, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved filtered retrieval examples to {RETRIEVAL_RESULTS_PATH}")


PDF-only chunks: 502
Chunks on PDF pages 56-62: 21
Transcript-only chunks: 14
Metadata filter checks passed
Saved filtered retrieval examples to retrieval_results.json


## Step 6: Build a source-grounded answer

The final answer uses the cross-encoder's direct comparison passage and a second, metadata-filtered retrieval over the EU AI Act's high-risk requirements pages. We rerank that filtered set with the same cross-encoder, then cite exact excerpts extracted from the corresponding PDF pages. The answer is a transparent template grounded in retrieved evidence, not an unverified generative-model response.

In [13]:
filtered_legal_candidates = search_chunks(
    TEST_QUERY,
    top_k=len(article_page_chunks),
    source=str(PDF_PATH),
    document_type="pdf",
    page_range=(56, 62),
)
filtered_legal_pairs = [
    (TEST_QUERY, candidate["text"])
    for candidate in filtered_legal_candidates
]
filtered_legal_scores = cross_encoder.predict(
    filtered_legal_pairs,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
)
if len(filtered_legal_scores) != len(filtered_legal_candidates):
    raise ValueError("The filtered cross-encoder did not score every legal chunk")
if not np.isfinite(filtered_legal_scores).all():
    raise ValueError("The filtered cross-encoder returned a non-finite score")

for candidate, score in zip(filtered_legal_candidates, filtered_legal_scores):
    candidate["filtered_cross_encoder_score"] = float(score)

filtered_legal_ranking = sorted(
    filtered_legal_candidates,
    key=lambda item: (-item["filtered_cross_encoder_score"], item["rank"]),
)
for rank, candidate in enumerate(filtered_legal_ranking, start=1):
    candidate["filtered_cross_encoder_rank"] = rank

def require_article_candidate(article_anchor: str, obligation_phrase: str) -> dict:
    matches = [
        candidate
        for candidate in filtered_legal_ranking
        if article_anchor in candidate["text"].lower()
        and obligation_phrase in candidate["text"].lower()
    ]
    if not matches:
        raise ValueError(f"Filtered results do not contain {article_anchor}")
    return matches[0]


selected_article_chunks = {
    "Article 9 risk management": require_article_candidate(
        "article 9 risk management system", "shall be established"
    ),
    "Article 10 data governance": require_article_candidate(
        "article 10 data and data governance", "data governance and management practices"
    ),
    "Article 14 human oversight": require_article_candidate(
        "article 14 human oversight", "high-risk ai systems shall be designed"
    ),
    "Article 16 provider duties": require_article_candidate(
        "article 16 obligations of providers of high-risk ai systems",
        "providers of high-risk ai systems shall:",
    ),
}

def extract_pdf_page_excerpt(
    page_number: int,
    anchor: str,
    *,
    context_before: int = 0,
    max_chars: int = 500,
) -> str:
    """Return a whitespace-normalized quote from one extracted PDF page."""
    page_document = next(
        (
            document
            for document in documents
            if document["metadata"]["source"] == str(PDF_PATH)
            and document["metadata"]["page"] == page_number
        ),
        None,
    )
    if page_document is None:
        raise ValueError(f"The PDF does not contain extracted page {page_number}")

    normalized_text = " ".join(page_document["text"].split())
    anchor_position = normalized_text.lower().find(anchor.lower())
    if anchor_position < 0:
        raise ValueError(f"Could not verify citation anchor on PDF page {page_number}")
    start = max(0, anchor_position - context_before)
    return normalized_text[start : start + max_chars]


citation_evidence = [
    {
        "citation": "[EU AI Act PDF, p. 8, recital 27]",
        "source": str(PDF_PATH),
        "page": 8,
        "reference": "Recital 27",
        "excerpt": extract_pdf_page_excerpt(
            8,
            "seven non-binding ethical principles",
            context_before=175,
            max_chars=720,
        ),
    },
    {
        "citation": "[EU AI Act PDF, p. 56, Article 9]",
        "source": str(PDF_PATH),
        "page": 56,
        "reference": "Article 9, Risk management system",
        "excerpt": extract_pdf_page_excerpt(
            56, "Article 9 Risk management system", max_chars=420
        ),
    },
    {
        "citation": "[EU AI Act PDF, p. 57, Article 10]",
        "source": str(PDF_PATH),
        "page": 57,
        "reference": "Article 10, Data and data governance",
        "excerpt": extract_pdf_page_excerpt(
            57, "Article 10 Data and data governance", max_chars=480
        ),
    },
    {
        "citation": "[EU AI Act PDF, p. 60, Article 14]",
        "source": str(PDF_PATH),
        "page": 60,
        "reference": "Article 14, Human oversight",
        "excerpt": extract_pdf_page_excerpt(
            60, "Article 14 Human oversight", max_chars=500
        ),
    },
    {
        "citation": "[EU AI Act PDF, p. 62, Article 16]",
        "source": str(PDF_PATH),
        "page": 62,
        "reference": "Article 16, Obligations of providers of high-risk AI systems",
        "excerpt": extract_pdf_page_excerpt(
            62, "Article 16 Obligations of providers of high-risk AI systems", max_chars=500
        ),
    },
]

final_answer = (
    "The question calls them ‘seven ethical requirements’, but recital 27 of the "
    "EU AI Act identifies the 2019 AI HLEG guidance as seven **non-binding ethical "
    "principles**: human agency and oversight; technical robustness and safety; "
    "privacy and data governance; transparency; diversity, non-discrimination and "
    "fairness; societal and environmental well-being; and accountability. The "
    "recital expressly distinguishes those guidelines from the Regulation’s "
    "legally binding requirements [EU AI Act PDF, p. 8, recital 27].\n\n"
    "By contrast, the Act imposes legal duties for high-risk AI systems. Examples "
    "in the supplied text include establishing and maintaining a risk-management "
    "system (Article 9, p. 56), applying data-governance practices to relevant "
    "training, validation and testing data (Article 10, p. 57), designing systems "
    "for effective human oversight (Article 14, p. 60), and provider duties such "
    "as ensuring compliance with Section 2 (Article 16, p. 62). These are examples, "
    "not an exhaustive list [EU AI Act PDF, pp. 56–57, 60, 62].\n\n"
    "In short: the seven principles are broad, non-binding ethical guidance; the "
    "Act’s high-risk requirements are specific legal obligations. They can address "
    "overlapping values, but they are not interchangeable checklists."
)

assert len(filtered_legal_candidates) == len(article_page_chunks)
assert all(
    result["metadata"]["source"] == str(PDF_PATH)
    and 56 <= result["metadata"]["page"] <= 62
    for result in filtered_legal_ranking
)
assert len(citation_evidence) == 5
assert all(item["excerpt"] and item["source"] == str(PDF_PATH) for item in citation_evidence)
assert "seven non-binding ethical principles" in citation_evidence[0]["excerpt"].lower()
assert "legally binding requirements" in citation_evidence[0]["excerpt"].lower()
assert "[EU AI Act PDF, p. 8, recital 27]" in final_answer
assert "[EU AI Act PDF, pp. 56–57, 60, 62]" in final_answer

print("Source-grounded answer:\n")
print(final_answer)
print("\nCitation evidence extracted from the cited PDF pages:")
for citation in citation_evidence:
    print(f"\n{citation['citation']} — {citation['reference']}")
    print(citation["excerpt"])

reranking_evidence["filtered_legal_retrieval"] = {
    "filters": {
        "source": str(PDF_PATH),
        "document_type": "pdf",
        "page_range": [56, 62],
    },
    "matching_chunk_count": len(article_page_chunks),
    "cross_encoder_ranking": filtered_legal_ranking,
}
reranking_evidence["rag_answer"] = {
    "text": final_answer,
    "citations": citation_evidence,
}
RERANKING_RESULTS_PATH.write_text(
    json.dumps(reranking_evidence, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\nSaved the final answer and cited evidence to {RERANKING_RESULTS_PATH}")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Source-grounded answer:

The question calls them ‘seven ethical requirements’, but recital 27 of the EU AI Act identifies the 2019 AI HLEG guidance as seven **non-binding ethical principles**: human agency and oversight; technical robustness and safety; privacy and data governance; transparency; diversity, non-discrimination and fairness; societal and environmental well-being; and accountability. The recital expressly distinguishes those guidelines from the Regulation’s legally binding requirements [EU AI Act PDF, p. 8, recital 27].

By contrast, the Act imposes legal duties for high-risk AI systems. Examples in the supplied text include establishing and maintaining a risk-management system (Article 9, p. 56), applying data-governance practices to relevant training, validation and testing data (Article 10, p. 57), designing systems for effective human oversight (Article 14, p. 60), and provider duties such as ensuring compliance with Section 2 (Article 16, p. 62). These are examples, n

## Step 7: Review answer quality and document a failure case

This is a qualitative source-to-claim review, not an automated relevance metric or independent human benchmark. Each citation quote is checked against the extracted PDF page. We also record where the cross-encoder failed to place the concrete Article 16 provider duties in its top five, despite the relevance-scored ranking promoting that evidence.

In [14]:
def extracted_pdf_page(page_number: int) -> str:
    page_document = next(
        (
            document
            for document in documents
            if document["metadata"]["source"] == str(PDF_PATH)
            and document["metadata"]["page"] == page_number
        ),
        None,
    )
    if page_document is None:
        raise ValueError(f"PDF page {page_number} was not loaded")
    return " ".join(page_document["text"].split())


for citation in citation_evidence:
    normalized_page = extracted_pdf_page(citation["page"])
    assert citation["excerpt"] in normalized_page

assert "seven non-binding ethical principles" in extracted_pdf_page(8).lower()
assert "legally binding requirements" in extracted_pdf_page(8).lower()
assert "shall be established, implemented, documented and maintained" in extracted_pdf_page(56).lower()
assert "data governance and management practices" in extracted_pdf_page(57).lower()
assert "effectively overseen by natural persons" in extracted_pdf_page(60).lower()
assert "providers of high-risk ai systems shall:" in extracted_pdf_page(62).lower()

article_16_position = summarize_evidence_position(
    article_16_chunk, "Article 16 high-risk AI provider duties"
)
manual_evaluation = {
    "method": "Qualitative source-to-claim review; not a numeric metric or independent human benchmark.",
    "checks": [
        {
            "claim": "The seven AI HLEG items are non-binding ethical principles, distinct from the Regulation's binding requirements.",
            "assessment": "Supported directly by recital 27 on PDF page 8.",
            "citation": "[EU AI Act PDF, p. 8, recital 27]",
        },
        {
            "claim": "The Act sets specific legal duties for high-risk systems.",
            "assessment": "Supported by the 'shall' provisions in Articles 9, 10, 14 and 16; the answer labels these as examples, not an exhaustive list.",
            "citations": [
                "[EU AI Act PDF, p. 56, Article 9]",
                "[EU AI Act PDF, p. 57, Article 10]",
                "[EU AI Act PDF, p. 60, Article 14]",
                "[EU AI Act PDF, p. 62, Article 16]",
            ],
        },
        {
            "claim": "Citations point to actual source evidence.",
            "assessment": "All five saved quotes were extracted from and matched against their cited PDF pages.",
        },
    ],
    "failure_case": {
        "description": "The cross-encoder did not place the concrete Article 16 provider duties in its top five, even though the Step 3 relevance scoring did.",
        "metadata": article_16_position["metadata"],
        "ranks": {
            "baseline_cosine": article_16_position["original_cosine_rank"],
            "step_3_relevance_scoring": article_16_position["relevance_scored_rank"],
            "cross_encoder": article_16_position["cross_encoder_rank"],
        },
        "cross_encoder_in_top_5": article_16_position["in_top_5"]["cross_encoder"],
        "observed_mitigation": "A PDF source/page filter over pages 56-62 retrieves statutory provisions separately; this is a targeted evidence search, not proof that the original top-20 ranking was complete.",
    },
    "limitations": [
        "The cross-encoder reranks only the cosine top 20 and cannot recover evidence that baseline retrieval omitted.",
        "Cross-encoder scores are raw ranking scores, not calibrated probabilities.",
        "The deterministic relevance scorer is a transparent keyword rubric, not a human relevance label or an LLM score.",
        "The podcast is a secondary transcript; the final legal-status distinction is grounded in the EU AI Act PDF.",
    ],
}

print("Qualitative answer and evidence review:")
for check in manual_evaluation["checks"]:
    print(f"- {check['claim']} {check['assessment']}")
print("\nDocumented failure case:")
print(manual_evaluation["failure_case"])
print("\nLimitations:")
for limitation in manual_evaluation["limitations"]:
    print(f"- {limitation}")

reranking_evidence["manual_evaluation"] = manual_evaluation
RERANKING_RESULTS_PATH.write_text(
    json.dumps(reranking_evidence, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\nSaved evaluation notes to {RERANKING_RESULTS_PATH}")


Qualitative answer and evidence review:
- The seven AI HLEG items are non-binding ethical principles, distinct from the Regulation's binding requirements. Supported directly by recital 27 on PDF page 8.
- The Act sets specific legal duties for high-risk systems. Supported by the 'shall' provisions in Articles 9, 10, 14 and 16; the answer labels these as examples, not an exhaustive list.
- Citations point to actual source evidence. All five saved quotes were extracted from and matched against their cited PDF pages.

Documented failure case:
{'description': 'The cross-encoder did not place the concrete Article 16 provider duties in its top five, even though the Step 3 relevance scoring did.', 'metadata': {'source': 'data/eu_ai_act.pdf', 'document_type': 'pdf', 'page': 62, 'chunk_index': 0}, 'ranks': {'baseline_cosine': 13, 'step_3_relevance_scoring': 4, 'cross_encoder': 12}, 'cross_encoder_in_top_5': False, 'observed_mitigation': 'A PDF source/page filter over pages 56-62 retrieves statu